# Property Location and Rental Price Analysis
### Statistical Analysis and Law of Large Numbers

**Course:** Probability and Statistics  
**Project Topic:** Property Location and Rental Price Analysis  
**Group Seminar Topic:** Law of Large Numbers in Real-Life Decision Making  

## Introduction

Rental prices vary quite a bit across different properties and cities. If we look at individual listings, it is hard to get a clear picture of what rent is typically like or how much it varies. This is where basic statistical analysis helps.

In this project, we use a real rental property dataset from Kaggle to study how rental prices are distributed across different cities and how property size relates to rent. We also connect this to our group seminar topic by demonstrating the Law of Large Numbers: as we take random samples of increasing size, the sample mean tends to settle closer to the mean of the full dataset.

## Problem Statement

Rental prices vary across properties and locations. Looking at one or two listings gives no real sense of the typical price or the spread of prices in the market. The project uses statistical analysis on real data to understand this variation and to show how the Law of Large Numbers applies to rental price data.

## Objectives

1. Understand the distribution of rental prices in the dataset.
2. Calculate all required statistical measures: Mean, Median, Mode, Variance, Standard Deviation, IQR, Range, Covariance, and Correlation.
3. Compare rental prices across different cities.
4. Study the relationship between property size and rent.
5. Calculate covariance and correlation between Size and Rent.
6. Demonstrate the Law of Large Numbers using random samples of increasing size.

## Dataset

We are using the **House Rent Prediction Dataset** from Kaggle, uploaded by iamsouravbanerjee.  
Source: https://www.kaggle.com/datasets/iamsouravbanerjee/house-rent-prediction-dataset

Each row represents one rental property listing. The main columns we use are:

- **Rent** - monthly rental price (main variable)
- **City** - city where the property is located (main location variable)
- **Size** - area of the property in square feet
- **BHK** - number of bedrooms, hall, and kitchen
- **Bathroom** - number of bathrooms
- **Furnishing Status** - whether the property is furnished, semi-furnished, or unfurnished

Not every column is used. We only use the ones that contribute to the analysis.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Load Dataset

In [ ]:
df = pd.read_csv("data/House_Rent_Dataset.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
df.head()

## 3. Understanding the Dataset

In [ ]:
df.info()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print("Cities in the dataset:")
print(df["City"].value_counts())

In [ ]:
print("Rent range: Rs.", df["Rent"].min(), "to Rs.", df["Rent"].max())
print("Size range:", df["Size"].min(), "to", df["Size"].max(), "sq. ft.")

## 4. Data Cleaning

The dataset has no missing values and no duplicate rows. All columns we need (Rent, Size, City, BHK, Bathroom) are already in numeric or string format with no conversion needed.

We will check for any obviously invalid values in Rent or Size.

In [ ]:
print("Rent <= 0:", (df["Rent"] <= 0).sum())
print("Size <= 0:", (df["Size"] <= 0).sum())

In [ ]:
# Check very small property sizes
print(df[df["Size"] < 50][["BHK", "Rent", "Size", "City"]])

There are a few listings with very small sizes. These could be data entry errors or extremely small studio spaces. Since they are a small number and we cannot confirm whether they are genuine or errors, we keep them in the dataset but note this as a limitation. The main analysis is not significantly affected by these rows.

## 5. Exploratory Data Analysis

In [ ]:
df[["Rent", "Size", "BHK", "Bathroom"]].describe().round(2)

In [ ]:
print("Furnishing Status:")
print(df["Furnishing Status"].value_counts())

### Rent Distribution

Let us look at how rental prices are distributed across the dataset.

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df["Rent"], bins=60, color="steelblue", edgecolor="white")
plt.title("Distribution of Rental Prices")
plt.xlabel("Rent (Rs.)")
plt.ylabel("Number of Properties")
plt.xlim(0, 300000)
plt.tight_layout()
plt.show()

The histogram is heavily right-skewed. Most properties have relatively low rents, but there are some very high-rent listings. This kind of skew means the mean will be pulled upward more than the median.

## 6. Statistical Analysis

We calculate the required statistical measures for Rent, which is the main numerical variable in this project.

In [ ]:
rent = df["Rent"]

# Mean: sum of all values / number of values
mean_rent = rent.mean()
print("Mean Rent: Rs.", round(mean_rent, 2))

In [ ]:
# Median: middle value when data is sorted
median_rent = rent.median()
print("Median Rent: Rs.", median_rent)

In [ ]:
# Mode: most frequently occurring value
mode_rent = rent.mode()
print("Mode of Rent:", mode_rent.tolist())

The mean rent is Rs. 34,993, while the median is Rs. 16,000. There is a noticeable gap between them. This is because of the right-skewed distribution: a small number of very expensive listings pull the mean upward, while the median stays closer to what a typical listing actually costs.

The mode is Rs. 15,000, which is the single most common rental price in the dataset.

In [ ]:
# Variance (sample variance, ddof=1)
var_rent = rent.var()
print("Variance of Rent:", round(var_rent, 2))

In [ ]:
# Standard Deviation (sample, ddof=1) - same unit as Rent
std_rent = rent.std()
print("Standard Deviation of Rent: Rs.", round(std_rent, 2))

In [ ]:
# IQR: spread of the middle 50% of the data
Q1 = rent.quantile(0.25)
Q3 = rent.quantile(0.75)
IQR = Q3 - Q1

print("Q1 (25th percentile): Rs.", Q1)
print("Q3 (75th percentile): Rs.", Q3)
print("IQR: Rs.", IQR)

In [ ]:
# Range
min_rent = rent.min()
max_rent = rent.max()
range_rent = max_rent - min_rent

print("Minimum Rent: Rs.", min_rent)
print("Maximum Rent: Rs.", max_rent)
print("Range: Rs.", range_rent)

The standard deviation (Rs. 78,106) is much larger than the mean (Rs. 34,993). This shows that rental prices vary a lot across properties. The IQR is Rs. 23,000, meaning the middle 50% of properties have rent between Rs. 10,000 and Rs. 33,000. The range is enormous (Rs. 34,98,800), largely because of a few extreme high-rent listings.

In [ ]:
# Outlier check using IQR rule
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(rent < lower_bound) | (rent > upper_bound)]
print("IQR lower bound: Rs.", lower_bound)
print("IQR upper bound: Rs.", upper_bound)
print("Properties outside this range:", len(outliers))

520 properties fall outside the IQR-based bounds. These are not necessarily data errors. High-rent properties in cities like Mumbai are real listings. We keep them in the dataset, but they do contribute to the large standard deviation and to the gap between mean and median.

### Summary Statistics Table

In [ ]:
size = df["Size"]
bhk = df["BHK"]
bathroom = df["Bathroom"]

summary = pd.DataFrame({
    "Variable": ["Rent (Rs.)", "Size (sq.ft.)", "BHK", "Bathroom"],
    "Mean":     [round(rent.mean(), 2), round(size.mean(), 2), round(bhk.mean(), 2), round(bathroom.mean(), 2)],
    "Median":   [rent.median(), size.median(), bhk.median(), bathroom.median()],
    "Mode":     [rent.mode()[0], size.mode()[0], bhk.mode()[0], bathroom.mode()[0]],
    "Std Dev":  [round(rent.std(), 2), round(size.std(), 2), round(bhk.std(), 2), round(bathroom.std(), 2)],
    "Min":      [rent.min(), size.min(), bhk.min(), bathroom.min()],
    "Max":      [rent.max(), size.max(), bhk.max(), bathroom.max()]
})

print(summary.to_string(index=False))

## 7. Location-wise Rental Analysis

The project topic is about property location and rental price. The main location variable is **City**. We compare rental prices across the six cities in the dataset.

In [ ]:
city_stats = df.groupby("City")["Rent"].agg(
    Count="count",
    Mean="mean",
    Median="median",
    Std="std"
).round(2)

city_stats = city_stats.sort_values("Mean", ascending=False)
print(city_stats)

In [ ]:
cities = city_stats.index.tolist()
avg_rents = city_stats["Mean"].tolist()

plt.figure(figsize=(9, 5))
plt.bar(cities, avg_rents, color="steelblue", edgecolor="white")
plt.title("Average Rent by City")
plt.xlabel("City")
plt.ylabel("Average Rent (Rs.)")
plt.tight_layout()
plt.show()

Mumbai has by far the highest average rent among the six cities, at around Rs. 85,321. Kolkata has the lowest average at Rs. 11,645. The differences across cities are quite large, which shows that location is an important factor when looking at rental prices.

The standard deviation for Mumbai and Bangalore is very high, which means there is a wide range of rental prices within those cities too.

## 8. Covariance and Correlation

We use **Size** and **Rent** for covariance and correlation. Both are numerical variables, which is required for these calculations to make sense. Using a categorical variable like City directly in a covariance formula would not be appropriate.

### Covariance

Covariance tells us whether two numerical variables tend to move in the same direction or in opposite directions. A positive covariance between Size and Rent means larger properties tend to have higher rents.

In [ ]:
cov_size_rent = np.cov(df["Size"], df["Rent"], ddof=1)[0][1]
print("Covariance (Size, Rent):", round(cov_size_rent, 2))

The covariance is positive (around 20,485,348). This tells us that Size and Rent move in the same direction: larger properties tend to have higher rents. However, the actual number is hard to interpret because covariance depends on the scale of both variables.

### Correlation

Correlation standardizes the covariance and gives a value between -1 and +1. This makes it much easier to say how strong or weak the relationship is, regardless of the units of measurement.

In [ ]:
corr_size_rent = df["Size"].corr(df["Rent"])
print("Correlation (Size, Rent):", round(corr_size_rent, 4))

The correlation is 0.4136. This is a moderate positive correlation. It means there is a relationship between property size and rent, but it is not very strong. Many other factors (city, furnishing, floor, etc.) also affect rent.

**Difference between covariance and correlation:**  
Covariance tells us the direction of the relationship but its value depends on the scale of both variables, so it is hard to compare across different variable pairs. Correlation divides the covariance by the product of both standard deviations, giving a standardized value between -1 and +1 that is easy to interpret regardless of units.

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(df["Size"], df["Rent"], alpha=0.3, color="steelblue", edgecolors="none", s=15)
plt.title("Size vs Rent")
plt.xlabel("Size (sq. ft.)")
plt.ylabel("Rent (Rs.)")
plt.ylim(0, 500000)
plt.tight_layout()
plt.show()

The scatter plot shows a general upward trend: larger properties tend to cost more. But there is also a lot of spread, meaning size alone does not fully explain rent. The correlation of 0.41 reflects this.

## 9. Law of Large Numbers

The **Law of Large Numbers** states that as the number of observations in a random sample increases, the sample mean tends to get closer to the true mean of the distribution.

For this project, the Kaggle dataset is not the entire population of rental properties in India. So we use the **mean of the complete available dataset** as our reference point, and demonstrate that random samples from this data show the LLN behavior.

In [ ]:
reference_mean = df["Rent"].mean()
print("Reference mean (complete dataset): Rs.", round(reference_mean, 2))
print("Total Rent observations:", len(df["Rent"]))

For this demonstration, the mean of the complete available dataset (Rs. 34,993.45) is treated as the reference mean. We will randomly sample different numbers of listings and track how the sample mean changes.

In [ ]:
random_state = 42
sample_sizes = [10, 25, 50, 100, 250, 500, 1000, 2000, 3000, 4000, 4746]

sample_means = []
diffs = []

for n in sample_sizes:
    sample = df["Rent"].sample(n=n, random_state=random_state)
    sm = sample.mean()
    sample_means.append(round(sm, 2))
    diffs.append(round(sm - reference_mean, 2))

lln_table = pd.DataFrame({
    "Sample Size": sample_sizes,
    "Sample Mean (Rs.)": sample_means,
    "Diff from Reference Mean (Rs.)": diffs
})

print(lln_table.to_string(index=False))

For small samples (n=10, 25), the sample mean can be quite different from the reference mean. As the sample size increases, the sample mean gets closer and stays closer to Rs. 34,993. This is the Law of Large Numbers in action.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(sample_sizes, sample_means, marker="o", color="steelblue", label="Sample Mean")
plt.axhline(y=reference_mean, color="red", linestyle="--",
            label="Reference Mean (Rs. " + str(round(reference_mean, 0)) + ")")
plt.title("Law of Large Numbers: Sample Mean vs Sample Size")
plt.xlabel("Sample Size")
plt.ylabel("Sample Mean Rental Price (Rs.)")
plt.legend()
plt.tight_layout()
plt.show()

The graph clearly shows that the sample mean starts far from the reference mean at small sample sizes and gradually converges toward it as the sample size increases. At n=4746 (the full dataset), the sample mean equals the reference mean exactly.

The sample mean does not always move strictly closer with every step since each sample is random. But the general trend of decreasing deviation is visible, which is consistent with the Law of Large Numbers.

**Why random sampling?**  
We use random sampling instead of just taking the first n rows because random sampling gives each property an equal chance of being selected. Taking the first rows would not represent the full variety of the dataset.

**Why a fixed random seed?**  
A fixed seed (42) means the random sampling gives the same result every time the notebook is run. This makes the results reproducible.

## 10. Key Findings

Based on the actual calculated results from the dataset:

1. The dataset contains **4,746 rental property listings** across 6 Indian cities, with 12 columns.
2. The **mean rent is Rs. 34,993** and the **median rent is Rs. 16,000**. The gap indicates that a few high-rent listings pull the average upward.
3. **Mumbai** has the highest average rent (Rs. 85,321) among the six cities. **Kolkata** has the lowest (Rs. 11,645). Location clearly has a large effect on rental prices.
4. The **standard deviation of rent is Rs. 78,106**, which is very large compared to the mean, reflecting the wide spread of rental prices.
5. **Size and Rent have a positive correlation of 0.4136**. Larger properties tend to have higher rents, but the relationship is moderate.
6. As the sample size increased in the LLN experiment, the **sample mean became more stable around the reference mean of Rs. 34,993**, demonstrating the Law of Large Numbers with real rental price data.

## 11. Limitations

- The dataset contains rental listings from one source and does not represent all rental properties across India.
- The data was collected at a specific point in time, so it may not reflect current market conditions.
- Rental prices depend on many factors (location within a city, floor, amenities, etc.) not all captured here.
- Correlation between Size and Rent does not imply that increasing size causes rent to increase. It is an observed relationship in this data.
- For the LLN demonstration, the mean of the complete available dataset is used as the reference mean, not the true population mean of all Indian rental properties.

## 12. Conclusion

In this project, we studied rental property data from 4,746 listings across 6 Indian cities. Using basic statistical measures, we found that rental prices are right-skewed with a mean significantly higher than the median, and that there is a lot of variation in prices. City is a strong factor in this variation, with Mumbai listings having an average rent more than seven times higher than Kolkata listings. Property size shows a moderate positive correlation (0.41) with rent.

The Law of Large Numbers was demonstrated using random samples of increasing size. The sample mean started far from the reference mean for small samples and steadily converged toward it as the sample size grew. This shows, using real rental data, why larger samples give more reliable estimates of the true average.